In [ ]:
! pip install requests

In [ ]:
! pip install pandas

In [ ]:
import pandas as pd

# Load csv file from lacal repository
df1 = pd.read_csv('Figure_S6C_NaCl_CrowdFRET_Processed.csv')
print(df1.head())  # Display the first few rows of the DataFrame

In [ ]:
# install matplotlib library
! pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# Define color gradient
def get_gradient_colors(start_color, end_color, n):
    cmap = mcolors.LinearSegmentedColormap.from_list("grad", [start_color, end_color])
    return [mcolors.rgb2hex(cmap(i / (n - 1))) for i in range(n)]

# Generate 9 colors (Isotonic + 8 NaCl levels)
colors = get_gradient_colors("#b3b3b3", "#49c1bb", 9)

# Apply rolling mean for smoothing
for cond in ['Isotonic'] + [f'NaCl_{i}mOsm/L' for i in range(50, 401, 50)]:
    df1[f'{cond}_Mean_smooth'] = df1[f'{cond}_Mean'].rolling(window=3, center=True).mean()
    df1[f'{cond}_SD_smooth'] = df1[f'{cond}_SD'].rolling(window=3, center=True).mean()

# Create the plot
plt.figure(figsize=(5,3))

# Plot each condition with gradient color
conditions = ['Isotonic'] + [f'NaCl_{i}mOsm/L' for i in range(50, 401, 50)]
for i, cond in enumerate(conditions):
    color = colors[i]
    plt.plot(df1['TIME'], df1[f'{cond}_Mean_smooth'], label=cond, color=color)
    plt.fill_between(
        df1['TIME'],
        df1[f'{cond}_Mean_smooth'] - df1[f'{cond}_SD_smooth'],
        df1[f'{cond}_Mean_smooth'] + df1[f'{cond}_SD_smooth'],
        color=color, alpha=0.2
    )

# Set x range
xticks = np.arange(0, 1250, 200)
plt.xticks(xticks)
plt.tick_params(axis='both', labelsize=12)

# Set y-axis range
yticks = np.arange(0.4, 1.9, 0.2)
plt.yticks(yticks)

# Adjust spines
ax = plt.gca()
ax.spines['left'].set_position(('outward', 5))
ax.spines['bottom'].set_position(('outward', 20))
ax.spines['top'].set_color('none')
ax.spines['right'].set_color('none')
ax.spines['bottom'].set_bounds(0, 1250)
ax.spines['left'].set_bounds(0.4, 1.8)

# Customize the plot
plt.title('Crowd FRET', fontsize=16, family='Arial')
plt.xlabel('Time (min)', fontsize=14, family='Arial', labelpad=5)
plt.ylabel('Normalized mCitrine/mCerulean\n(525/475 nm)', fontsize=14, family='Arial', labelpad=5)
plt.grid(False)

# Legend
plt.legend(
    frameon=False,
    loc='center left',
    bbox_to_anchor=(1, 0.5),
    prop={'family': 'Arial', 'size': 12}
)

plt.subplots_adjust(right=0.75)

# Save the plot
plt.savefig('Figure_S6C_CrowdFRET_NaCl_gradient.pdf', format='pdf', bbox_inches='tight')
plt.show()